# 01d — Destriping relevance on the skin region

`01c` validated the destriping globally (whole-image streaking). But the model's target is
EI **on facial skin only**, so what matters here is does destriping help *on the
skin*? Additionally, does it remove the stripe **without** corrupting the erythema signal?
This became answerable only once the face-skin masks existed (`02`).

**What:** compare raw vs destriped EI over skin pixels (mask==1). **How:** (1) a column-streaking
metric on skin across all 306 images; (2) a visual check that the *removed* component (raw − destriped)
is stripe, not anatomy. **Why:** confirms the destriped target the model learns is de-striped yet
faithful. Images shown only for the display-permitted subjects (p012/p019/p027).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config

PROCESSED = (Path(config.__file__).resolve().parent / config.LOCAL_PROCESSED_DIR).resolve()
RAW_DIR   = PROCESSED / "ei_maps"                # raw Dawson EI
DES_DIR   = PROCESSED / "ei_maps_destriped"      # destriped EI (the model target)
MASKS_DIR = PROCESSED / "masks"                  # binary face-skin masks

manifest   = pd.read_csv(PROCESSED / "manifest.csv")
DISCLOSURE = ["p012", "p019", "p027"]            # only subjects permitted for display
print("images:", len(manifest), "| splits:", manifest.split.value_counts().to_dict())

## 1. Quantitative — skin column-streaking, raw vs destriped (all 306)

The push-broom stripe is a per-column offset, so it appears as high-frequency wiggle across the
per-column means of the skin. This is the **same streaking metric as `01b`**, applied here to skin
pixels only (mask==1) instead of the whole image.

For each image, let **`mu_j`** = the mean EI of the skin pixels in column *j* (one value per column,
scanned across the width of the face). The metric is the **second difference**

    | mu_j − (mu_{j−1} + mu_{j+1}) / 2 |

averaged over columns and divided by the p1–p99 range of the column means (so it is scale-free;
lower = less striping). Intuition: real anatomy varies **smoothly** across columns, so a column sits
close to the average of its two neighbours → small value; a **stripe** makes a column jump relative
to its neighbours → large value. The second difference therefore isolates the high-frequency stripe
from the smooth facial trend.

**How to read the output.** The number that matters is the **mean metric**, which drops on every
split. The per-image `destriped < raw` figure is just a simple count of how many images improved —
**not** a success/failure count: every image *is* destriped. The offset is fitted to the
**whole-image** column stripe, while on skin the residual is value-dependent and the metric is
noisy, so on a minority of images the skin metric barely moves either way. Faithfulness is
established by the visual in §2, not by this count.

In [ ]:
def skin_streaking(ei, mask):
    """Column-mean high-frequency streaking over skin pixels, normalised by dynamic range."""
    cnt = mask.sum(0)
    valid = cnt >= 10
    cm = np.full(ei.shape[1], np.nan)
    cm[valid] = (ei * mask).sum(0)[valid] / cnt[valid]
    d = np.abs(cm[1:-1] - (cm[:-2] + cm[2:]) / 2)
    ok = valid[1:-1] & valid[:-2] & valid[2:]
    if ok.sum() == 0:
        return np.nan
    dyn = np.nanpercentile(cm[valid], 99) - np.nanpercentile(cm[valid], 1)
    return float(np.nanmean(d[ok]) / dyn) if dyn > 0 else np.nan


rows = []
for r in manifest.itertuples(index=False):
    stem = f"{r.subject_id}_{r.pose}_{r.view}"
    m   = np.load(MASKS_DIR / f"{stem}.npy").astype(bool)
    raw = np.load(RAW_DIR / f"{stem}.npy")
    des = np.load(DES_DIR / f"{stem}.npy")
    rows.append({"split": r.split,
                 "raw": skin_streaking(raw, m),
                 "destriped": skin_streaking(des, m)})
d = pd.DataFrame(rows)

print(f"Skin column-streaking (lower = less stripe), mean over {len(d)} images:")
print(f"  raw       = {d.raw.mean():.4f}")
print(f"  destriped = {d.destriped.mean():.4f}")
print(f"  reduction = {(1 - d.destriped.mean() / d.raw.mean()) * 100:.1f}%")
print(f"  destriped < raw in {(d.destriped < d.raw).sum()}/{len(d)} images")
print()
print("by split (mean):")
print(d.groupby("split")[["raw", "destriped"]].mean().round(4).to_string())

## 2. Visual — the removed component is stripe, not anatomy (disclosure subjects)

The key correctness check. Columns: raw EI (skin) · destriped EI (skin) · **removed = raw − destriped**
· per-column skin mean. If destriping is faithful, the *removed* panel is a pure vertical-stripe
pattern with **no visible face** (it took out the stripe, not the erythema), and the destriped
column-mean profile is smoother than the raw one while following the same anatomical trend.

In [ ]:
demo = [f"{s}_neutral_front" for s in DISCLOSURE]
fig, axes = plt.subplots(len(demo), 4, figsize=(18, 4.6 * len(demo)))
for i, stem in enumerate(demo):
    raw = np.load(RAW_DIR / f"{stem}.npy")
    des = np.load(DES_DIR / f"{stem}.npy")
    m   = np.load(MASKS_DIR / f"{stem}.npy").astype(bool)
    removed = raw - des
    sk = lambda a: np.where(m, a, np.nan)
    lo, hi = np.nanpercentile(sk(raw), 2), np.nanpercentile(sk(raw), 98)

    axes[i, 0].imshow(sk(raw), cmap="magma", vmin=lo, vmax=hi)
    axes[i, 0].set_title(f"{stem}\nraw EI (skin)", fontsize=9)
    axes[i, 1].imshow(sk(des), cmap="magma", vmin=lo, vmax=hi)
    axes[i, 1].set_title("destriped EI (skin)", fontsize=9)
    axes[i, 2].imshow(removed, cmap="coolwarm", vmin=-3, vmax=3)
    axes[i, 2].set_title("removed = raw - destriped", fontsize=9)

    cnt = m.sum(0); v = cnt >= 10
    cmr = np.where(v, (raw * m).sum(0) / np.maximum(cnt, 1), np.nan)
    cmd = np.where(v, (des * m).sum(0) / np.maximum(cnt, 1), np.nan)
    axes[i, 3].plot(cmr, lw=0.6, label="raw")
    axes[i, 3].plot(cmd, lw=0.6, label="destriped")
    axes[i, 3].set_title("per-column skin mean", fontsize=9)
    axes[i, 3].set_xlabel("column"); axes[i, 3].legend(fontsize=8)
    for k in range(3):
        axes[i, k].axis("off")
plt.suptitle("Destriping on the skin region: the removed component is stripe, not anatomy", y=1.001)
plt.tight_layout(); plt.show()

## Conclusion

- **Faithful (the decisive check):** the removed component (raw − destriped) is a pure vertical-stripe
  pattern with no visible facial structure — destriping subtracts the stripe, not the erythema signal.
- **Effective on skin:** the per-column skin-mean profile is smoothed (the sharp column-to-column
  spikes are removed) while the anatomical trend is preserved.
- **Faint residual remains** (value-dependent part; no simple per-column method removes it fully).
  It is zero-mean target noise; because the RGB input is clean the model cannot hallucinate stripes,
  so the residual only adds a small noise floor to the target (a modest ceiling on SSIM).
- **Verdict:** destriping is relevant and safe for the skin-only target — de-striped yet faithful.